# Cleaned analysis notebook

This copy preserves the original analysis/calculation cells. Personal machine paths have been replaced with project-relative paths.
Place input workbooks and data in `data/`; generated files are written to `outputs/`. Create these folders before running.
Saved cell outputs were cleared so results are regenerated when the notebook is run.


## Input and output paths

This notebook uses paths relative to the repository root. Place input files under `data/` and generated results under `outputs/`. No Google Drive mount is required.

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd

FAILURE_FILE = "data/Original_dice_all_fold.xlsx"

GT_DIR = "data/masks"
PRED_DIR = "data/predictions"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
pred_files = glob.glob(os.path.join(PRED_DIR, "*.nii"))

In [ ]:
dice_df = pd.read_excel(FAILURE_FILE)

# Calculate an overall 'Dice' score by averaging the 'Class_X' columns
class_columns = [col for col in dice_df.columns if col.startswith('Class_')]
dice_df['Dice'] = dice_df[class_columns].mean(axis=1)

thresholds = [0.60, 0.70, 0.80]

for threshold in thresholds:

    failure_rows = dice_df[dice_df["Dice"] < threshold].copy()

    failure_cases = (
        failure_rows["Case"]
        .drop_duplicates()
        .sort_values()
    )

    print(f"\nDSC < {threshold}")
    print("Number of failure cases:", len(failure_cases))
    print("Failure rate:", len(failure_cases) / dice_df["Case"].nunique() * 100)

Calcualte Median / Minimum / 10th percentile /Maximum (Vertebrae + IVD)

In [ ]:
import numpy as np

dice = df[class_cols].stack().dropna()

print("N:", len(dice))
print("Mean:", dice.mean())
print("SD:", dice.std())
print("Median:", dice.median())
print("Minimum:", dice.min())
print("10th percentile:", np.percentile(dice, 10))
print("25th percentile:", np.percentile(dice, 25))
print("75th percentile:", np.percentile(dice, 75))
print("90th percentile:", np.percentile(dice, 90))
print("Maximum:", dice.max())

In [ ]:
quantiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

for q in quantiles:
    print(f"{q}th percentile:",
          np.percentile(dice, q))

Empirical Cumulative Distribution of Dice Scores

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Extract valid structure-level Dice scores
dice = np.sort(df[class_cols].stack().dropna().values)

# Calculate empirical cumulative proportions
y = np.arange(1, len(dice) + 1) / len(dice)

# Plot ECDF
plt.figure(figsize=(7, 5))
plt.plot(dice, y)
plt.xlabel("Dice Similarity Coefficient")
plt.ylabel("Cumulative Proportion")
plt.title("C. Empirical Cumulative Distribution of Dice Scores")
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save before displaying
plt.savefig(
    "outputs/ECD_of_Dice_Score.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Number of valid Dice observations:", len(dice))


In [ ]:
import numpy as np

# --------------------------------------------------
# Vertebrae: Class 1-5
# IVDs: Class 6-11
# --------------------------------------------------

vertebra_cols = [f'Class_{i}' for i in range(1, 6)]
ivd_cols = [f'Class_{i}' for i in range(6, 12)]


# --------------------------------------------------
# Threshold-free distribution statistics
# --------------------------------------------------

def distribution_stats(df, columns, name):

    dice = df[columns].stack().dropna()

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("N:", len(dice))
    print("Mean:", dice.mean())
    print("SD:", dice.std())
    print("Median:", dice.median())
    print("Minimum:", dice.min())
    print("10th percentile:", np.percentile(dice, 10))
    print("25th percentile:", np.percentile(dice, 25))
    print("75th percentile:", np.percentile(dice, 75))
    print("90th percentile:", np.percentile(dice, 90))
    print("Maximum:", dice.max())


# --------------------------------------------------
# Vertebrae distribution
# --------------------------------------------------

distribution_stats(
    df,
    vertebra_cols,
    "VERTEBRAE (Class 1-5)"
)


# --------------------------------------------------
# IVD distribution
# --------------------------------------------------

distribution_stats(
    df,
    ivd_cols,
    "IVDs (Class 6-11)"
)

In [ ]:
import numpy as np

# Class 1-5 = Vertebrae
vertebra_cols = [f'Class_{i}' for i in range(1, 6)]

# Class 6-11 = IVDs
ivd_cols = [f'Class_{i}' for i in range(6, 12)]


quantiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]


# -------------------------------
# Vertebrae
# -------------------------------

vertebra_dice = df[vertebra_cols].stack().dropna().values

print("=" * 50)
print("VERTEBRAE (Class 1-5)")
print("=" * 50)

for q in quantiles:
    print(f"{q}th percentile:",
          np.percentile(vertebra_dice, q))


# -------------------------------
# IVDs
# -------------------------------

ivd_dice = df[ivd_cols].stack().dropna().values

print("\n" + "=" * 50)
print("IVDs (Class 6-11)")
print("=" * 50)

for q in quantiles:
    print(f"{q}th percentile:",
          np.percentile(ivd_dice, q))

Empirical_Cumulative_Distribution_of_Vertebral_Dice_Scores

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Class 1-5 = Vertebrae
vertebra_cols = [f'Class_{i}' for i in range(1, 6)]

vertebra_dice = np.sort(
    df[vertebra_cols].stack().dropna().values
)

y_vertebra = np.arange(1, len(vertebra_dice) + 1) / len(vertebra_dice)

plt.figure(figsize=(7, 5))

plt.plot(vertebra_dice, y_vertebra)

plt.xlabel("Dice Similarity Coefficient")
plt.ylabel("Cumulative Proportion of Vertebral Scores")
plt.title("A. Empirical Cumulative Distribution of Vertebral Dice Scores")

plt.grid(True, alpha=0.3)
plt.tight_layout()


plt.savefig(
    "outputs/Empirical_Cumulative_Distribution_of_Vertebral_Dice_Scores.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

Empirical_Cumulative_Distribution_of_IVD_Dice_Score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Class 6-11 = IVDs
ivd_cols = [f'Class_{i}' for i in range(6, 12)]

ivd_dice = np.sort(
    df[ivd_cols].stack().dropna().values
)

y_ivd = np.arange(1, len(ivd_dice) + 1) / len(ivd_dice)

plt.figure(figsize=(7, 5))

plt.plot(ivd_dice, y_ivd)

plt.xlabel("Dice Similarity Coefficient")
plt.ylabel("Cumulative Proportion for Disc Scores")
plt.title("B. Empirical Cumulative Distribution of IVD Dice Scores")

plt.grid(True, alpha=0.3)
plt.tight_layout()


plt.savefig(
    "outputs/Empirical_Cumulative_Distribution_of_IVD_Dice_Scores.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# **Patient-clustered 95% CI**
Calculate Patient-Clustred 95% CI for vertebrae

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import t

#df = pd.read_excel("your_file.xlsx")

# Corrected: Use 'Class_X' names for vertebrae columns
vertebra_cols = [f'Class_{i}' for i in range(1, 6)]

# Patient-level mean
patient_mean = df[vertebra_cols].mean(axis=1)

n = patient_mean.notna().sum()
mean_dice = patient_mean.mean()
sd_patient = patient_mean.std(ddof=1)

se = sd_patient / np.sqrt(n)
tcrit = t.ppf(0.975, df=n-1)

ci_low = mean_dice - tcrit * se
ci_high = mean_dice + tcrit * se

print("N patients:", n)
print("Mean:", mean_dice)
print("95% CI:", ci_low, ci_high)

Calculate Patient-Clustred 95% CI for IVD

In [ ]:
# Corrected: Use 'Class_X' names for IVD columns
ivd_cols = [f'Class_{i}' for i in range(6, 12)]

patient_mean = df[ivd_cols].mean(axis=1)

n = patient_mean.notna().sum()
mean_dice = patient_mean.mean()
sd_patient = patient_mean.std(ddof=1)

se = sd_patient / np.sqrt(n)
tcrit = t.ppf(0.975, df=n-1)

ci_low = mean_dice - tcrit * se
ci_high = mean_dice + tcrit * se

print("N patients:", n)
print("Mean:", mean_dice)
print("95% CI:", ci_low, ci_high)

# **Structure-level analysis**
Calculate mean, SD, and number of cases for each vertebral level

In [ ]:
# First, transform the DataFrame to a long format for vertebrae
vertebra_df_long = df.melt(
    id_vars=['Case'],
    value_vars=vertebra_cols, # Use the already defined vertebra_cols (Class_1 to Class_5)
    var_name='class_name',
    value_name='dice'
)

# Extract the vertebra level from the class name (e.g., 'Class_1' -> '1')
vertebra_df_long['vertebra_level'] = vertebra_df_long['class_name'].str.extract('Class_(\d+)').astype(int)

vertebra_stats = (
    vertebra_df_long.groupby("vertebra_level")["dice"]
      .agg(["mean", "std", "count"])
      .reset_index()
)

# Create Mean ± SD column
vertebra_stats["Dice (Mean ± SD)"] = (
    vertebra_stats["mean"].round(3).astype(str)
    + " \u00B1 " # Unicode for plus-minus sign
    + vertebra_stats["std"].round(3).astype(str)
)

# Keep required columns
vertebra_stats = vertebra_stats[
    ["vertebra_level", "Dice (Mean \u00B1 SD)", "count"]
]

# Rename count
vertebra_stats = vertebra_stats.rename(
    columns={"count": "Number of Cases"}
)

print(vertebra_stats)

Calculate mean, SD, and number of cases for each IVD level

In [ ]:
# First, transform the DataFrame to a long format for IVD
disc_df_long = df.melt(
    id_vars=['Case'],
    value_vars=ivd_cols, # Corrected: Use ivd_cols
    var_name='class_name',
    value_name='dice'
)

# Extract the vertebra level from the class name (e.g., 'Class_1' -> '1')
disc_df_long['intervertebral disc_level'] = disc_df_long['class_name'].str.extract('Class_(\d+)').astype(int)

disc_stats = (
    disc_df_long.groupby("intervertebral disc_level")["dice"]
      .agg(["mean", "std", "count"])
      .reset_index()
)

# Create Mean ± SD column
disc_stats["Dice (Mean ± SD)"] = (
    disc_stats["mean"].round(3).astype(str)
    + " \u00B1 " # Unicode for plus-minus sign
    + disc_stats["std"].round(3).astype(str)
)

# Keep required columns
disc_stats = disc_stats[
    ["intervertebral disc_level", "Dice (Mean \u00B1 SD)", "count"] # Corrected: Use 'intervertebral disc_level'
]

# Rename count
disc_stats = disc_stats.rename(
    columns={"count": "Number of Cases"}
)

print(disc_stats)

# **Box-and-whisker plot (boxplot)**
DSC distribution across intervertebral levels

In [ ]:
import matplotlib.pyplot as plt

# Map class numbers to anatomical levels
level_mapping = {
    1: "L5",
    2: "L4",
    3: "L3",
    4: "L2",
    5: "L1"
}

plot_df = vertebra_df_long.copy()
plot_df["Level"] = plot_df["vertebra_level"].map(level_mapping)

# Keep only vertebral levels
plot_df = plot_df.dropna(subset=["Level", "dice"])

# Data in anatomical order
levels = ["L1", "L2", "L3", "L4", "L5"]
data = [
    plot_df.loc[plot_df["Level"] == level, "dice"].values
    for level in levels
]

# Create boxplot
plt.figure(figsize=(8, 6))

plt.boxplot(
    data,
    labels=levels,
    patch_artist=False,
    showfliers=False
)

# Overlay individual DSC measurements
for i, values in enumerate(data, start=1):
    # small horizontal jitter
    import numpy as np
    jitter = np.random.uniform(-0.08, 0.08, size=len(values))

    plt.scatter(
        np.full(len(values), i) + jitter,
        values,
        alpha=0.35,
        s=18
    )

plt.xlabel("Vertebral Level")
plt.ylabel("Dice Similarity Coefficient (DSC)")
plt.title("DSC Distribution Across Vertebral Levels")

plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

DSC distribution across intervertebral levels

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd # Import pandas

# Map class numbers to anatomical levels
level_mapping = {
    6: "L5/S1",
    7: "L4/L5",
    8: "L3/L4",
    9: "L2/L3",
    10: "L1/L2",
    11: "T12/L1"
}

# First, transform the DataFrame to a long format for IVDs
disc_df_long = df.melt(
    id_vars=['Case'],
    value_vars=ivd_cols, # Use the already defined ivd_cols (Class_6 to Class_11)
    var_name='class_name',
    value_name='dice'
)

# Extract the IVD level from the class name (e.g., 'Class_6' -> '6')
disc_df_long['Intervertebral disc_level'] = disc_df_long['class_name'].str.extract(r'Class_(\d+)').astype(int)


plot_df = disc_df_long.copy()
plot_df["Level"] = plot_df["Intervertebral disc_level"].map(level_mapping)

# Keep only IVD levels
plot_df = plot_df.dropna(subset=["Level", "dice"])

# Data in anatomical order
levels = ["T12/L1", "L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]
data = [
    plot_df.loc[plot_df["Level"] == level, "dice"].values
    for level in levels
]

# Create boxplot
plt.figure(figsize=(8, 6))

plt.boxplot(
    data,
    labels=levels,
    patch_artist=False,
    showfliers=False
)

# Overlay individual DSC measurements
for i, values in enumerate(data, start=1):
    # small horizontal jitter
    import numpy as np
    jitter = np.random.uniform(-0.08, 0.08, size=len(values))

    plt.scatter(
        np.full(len(values), i) + jitter,
        values,
        alpha=0.35,
        s=18
    )

plt.xlabel("Intervertebral disc Level")
plt.ylabel("Dice Similarity Coefficient (DSC)")
plt.title("DSC Distribution Across Intervertebral Levels")

plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()